# 02. スナップショットとタイムトラベル（Spark）

Iceberg は書き込みのたびにスナップショット（その時点のテーブルの状態）を残します。
スナップショットを指定すると、**過去の時点のテーブルをそのまま読める**（タイムトラベル）ほか、過去の状態に**ロールバック**することもできます。

- テーブル: `handson.tt_spark`

## 準備: SparkSession を作る

接続設定は `spark-defaults.conf` にあるので、ここでは何も指定しません（01 と同じ）。

In [ ]:
from decimal import Decimal
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("02_time_travel").getOrCreate()

def sql(query):
    """SQL を実行し、結果があれば表示する"""
    df = spark.sql(query)
    if df.columns:
        df.show(truncate=False)

## 1. 3回に分けて書き込む

1回目と2回目で INSERT、3回目で DELETE します。書き込みごとにスナップショットが1つずつできます。

In [ ]:
sql("DROP TABLE IF EXISTS handson.tt_spark PURGE")
sql("CREATE TABLE handson.tt_spark (trip_id BIGINT, vendor STRING, fare DECIMAL(10, 2)) USING iceberg")

sql("INSERT INTO handson.tt_spark VALUES (1, 'A', 12.50), (2, 'B', 30.00)")   # 1回目
sql("INSERT INTO handson.tt_spark VALUES (3, 'A', 8.00), (4, 'C', 22.00)")    # 2回目
sql("DELETE FROM handson.tt_spark WHERE vendor = 'A'")                        # 3回目

sql("SELECT * FROM handson.tt_spark ORDER BY trip_id")

## 2. スナップショットの一覧

メタデータテーブル `snapshots` で、スナップショットの ID、作られた時刻、操作の種類を確認します。
後で使うので、ID と時刻を Python の変数に取っておきます。

In [ ]:
sql("SELECT committed_at, snapshot_id, parent_id, operation FROM handson.tt_spark.snapshots ORDER BY committed_at")

snapshots = spark.sql("SELECT snapshot_id, committed_at FROM handson.tt_spark.snapshots ORDER BY committed_at").collect()
first_id, second_id, third_id = [row.snapshot_id for row in snapshots]
first_at = snapshots[0].committed_at
print("1回目:", first_id, first_at)
print("2回目:", second_id)
print("3回目:", third_id)

## 3. タイムトラベル

### スナップショット ID を指定する

`VERSION AS OF <スナップショット ID>` で、その時点のテーブルを読みます。
3回目の DELETE で消えた vendor A の行が、2回目の時点ではまだ残っています。

In [ ]:
print("1回目の時点")
sql(f"SELECT * FROM handson.tt_spark VERSION AS OF {first_id} ORDER BY trip_id")
print("2回目の時点")
sql(f"SELECT * FROM handson.tt_spark VERSION AS OF {second_id} ORDER BY trip_id")

### 時刻を指定する

`TIMESTAMP AS OF '<時刻>'` は、指定した時刻の時点で最新だったスナップショットを読みます。

In [ ]:
sql(f"SELECT * FROM handson.tt_spark TIMESTAMP AS OF '{first_at}' ORDER BY trip_id")

### DataFrame API から

SQL を使わず、読み込みオプションで指定することもできます。

In [ ]:
spark.read.option("versionAsOf", second_id).table("handson.tt_spark").orderBy("trip_id").show()

## 4. タグで名前を付ける

スナップショット ID は覚えにくいので、**タグ**で名前を付けられます（Git のタグと同じ考え方）。
タグを付けたスナップショットは、07 のスナップショット失効でも消されずに残ります。

In [ ]:
sql(f"ALTER TABLE lakehouse.handson.tt_spark CREATE TAG `before_delete` AS OF VERSION {second_id}")
sql("SELECT name, type, snapshot_id FROM handson.tt_spark.refs")
sql("SELECT * FROM handson.tt_spark VERSION AS OF 'before_delete' ORDER BY trip_id")

## 5. ロールバック

`rollback_to_snapshot` プロシージャで、テーブルの現在の状態を過去のスナップショットに戻します。
DELETE する前（2回目）に戻すと、消した行が復活します。

In [ ]:
sql(f"CALL lakehouse.system.rollback_to_snapshot('handson.tt_spark', {second_id})")
sql("SELECT * FROM handson.tt_spark ORDER BY trip_id")

ロールバックしても、3回目のスナップショットは消えません。
`history` を見ると、3回目は `is_current_ancestor = false`（現在の状態の祖先ではない）になっています。

In [ ]:
sql("SELECT made_current_at, snapshot_id, is_current_ancestor FROM handson.tt_spark.history ORDER BY made_current_at")

## まとめ

- 書き込みのたびにスナップショットができ、`VERSION AS OF` / `TIMESTAMP AS OF` で過去の状態を読める
- タグでスナップショットに名前を付けられる
- `rollback_to_snapshot` で現在の状態を過去に戻せる。戻した後もスナップショット自体は残る
- 古いスナップショットは溜まり続けるので、07 のメンテナンスで整理する